# XAUUSD/XAGUSD Trading LLM Fine-Tuning on Colab

This notebook fine-tunes a small LLM (Mistral-7B or Llama-2-7B) using PEFT (LoRA) on spike-labeled trading data.

**Steps:**
1. Install dependencies
2. Upload or download JSONL training/test files
3. Load base model with 4-bit quantization
4. Fine-tune using LoRA adapters
5. Evaluate on test set
6. Download adapters for inference

**Requirements:**
- Google Colab with GPU (T4 or V100 recommended)
- ~10-15 GB free storage
- Training time: 1-2 hours depending on dataset size

## 1. Setup: Install dependencies

In [ ]:
# Install required packages
!pip install -q torch transformers peft bitsandbytes accelerate datasets scikit-learn

import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU info: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA version: {torch.version.cuda}")

## 2. Upload training data

Follow these steps:
1. Download JSONL files from your local machine: `data/datasets/train_xauusd.jsonl` and `test_xauusd.jsonl`
2. Run the cell below and upload the files when prompted

Alternatively, set `USE_SAMPLE_DATA=True` below to create a demo dataset.

In [ ]:
import json
import os
from pathlib import Path

# Create data directory
Path("data/datasets").mkdir(parents=True, exist_ok=True)

USE_SAMPLE_DATA = True  # Set to False if you upload your own data

if USE_SAMPLE_DATA:
    # Create a small demo dataset for testing
    demo_train = [
        {
            "prompt": "Recent XAUUSD price action:\n  Bar 1: O=2100.0, H=2105.0, L=2098.0, C=2103.0\n  Bar 2: O=2103.0, H=2110.0, L=2102.0, C=2108.0\nBased on this pattern, should we BUY, SELL, or HOLD? Respond with: BUY, SELL, or HOLD.",
            "response": "BUY"
        },
        {
            "prompt": "Recent XAUUSD price action:\n  Bar 1: O=2108.0, H=2112.0, L=2106.0, C=2110.0\nBased on this pattern, should we BUY, SELL, or HOLD? Respond with: BUY, SELL, or HOLD.",
            "response": "HOLD"
        },
        {
            "prompt": "Recent XAUUSD price action:\n  Bar 1: O=2110.0, H=2108.0, L=2100.0, C=2102.0\nBased on this pattern, should we BUY, SELL, or HOLD? Respond with: BUY, SELL, or HOLD.",
            "response": "SELL"
        },
    ]
    
    demo_test = [
        {
            "prompt": "Recent XAUUSD price action:\n  Bar 1: O=2102.0, H=2115.0, L=2100.0, C=2112.0\nBased on this pattern, should we BUY, SELL, or HOLD? Respond with: BUY, SELL, or HOLD.",
            "response": "BUY"
        },
    ]
    
    with open("data/datasets/train_xauusd.jsonl", "w") as f:
        for ex in demo_train:
            f.write(json.dumps(ex) + "\n")
    
    with open("data/datasets/test_xauusd.jsonl", "w") as f:
        for ex in demo_test:
            f.write(json.dumps(ex) + "\n")
    
    print("Created demo training data (3 examples) and test data (1 example)")
else:
    from google.colab import files
    print("Upload your JSONL files:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".jsonl"):
            os.rename(filename, f"data/datasets/{filename}")
            print(f"Saved {filename} to data/datasets/")

## 3. Load data and split if needed

In [ ]:
import json
import pandas as pd

def load_jsonl(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Load training and test data
train_data = load_jsonl("data/datasets/train_xauusd.jsonl")
test_data = load_jsonl("data/datasets/test_xauusd.jsonl")

print(f"Training examples: {len(train_data)}")
print(f"Test examples: {len(test_data)}")

# Show example
if train_data:
    print("\nExample training sample:")
    print(f"  Prompt: {train_data[0]['prompt'][:100]}...")
    print(f"  Response: {train_data[0]['response']}")

## 4. Load base model and tokenizer with 4-bit quantization

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Model choice: use a smaller model for demos, larger for production
# Options: "mistralai/Mistral-7B-Instruct-v0.1", "meta-llama/Llama-2-7b-chat-hf", "meta-llama/Llama-2-3b-hf"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
# For smaller VRAM: use MODEL_NAME = "meta-llama/Llama-2-3b-hf"

print(f"Loading {MODEL_NAME}...")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model loaded successfully (quantized to 4-bit)")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Setup PEFT (LoRA) configuration

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA config
peft_config = LoraConfig(
    r=8,                           # LoRA rank
    lora_alpha=16,                 # LoRA scaling
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # modules to apply LoRA to
)

# Get PEFT model
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print(f"\nLoRA setup complete")

## 6. Prepare data for training

In [ ]:
from datasets import Dataset

def format_for_training(examples):
    """Format examples as prompt + response."""
    texts = []
    for ex in examples:
        prompt = ex["prompt"]
        response = ex["response"]
        # Format: prompt + response, separated by a newline
        text = f"{prompt}\n{response}"
        texts.append(text)
    return {"text": texts}

# Create HF datasets
train_dataset = Dataset.from_dict({"text": [f"{ex['prompt']}\n{ex['response']}" for ex in train_data]})
test_dataset = Dataset.from_dict({"text": [f"{ex['prompt']}\n{ex['response']}" for ex in test_data]})

print(f"Training dataset: {len(train_dataset)} examples")
print(f"Test dataset: {len(test_dataset)} examples")

## 7. Tokenize data

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print(f"Tokenized training dataset: {len(tokenized_train)} examples")
print(f"Tokenized test dataset: {len(tokenized_test)} examples")
print(f"Sample token ids (first 20): {tokenized_train[0]['input_ids'][:20]}")

## 8. Train using Trainer

In [ ]:
from transformers import Trainer, TrainingArguments
import warnings
warnings.filterwarnings('ignore')

training_args = TrainingArguments(
    output_dir="./models/checkpoints/xauusd_lora",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,  # Adjust based on GPU memory
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    save_steps=100,
    eval_steps=50,
    logging_steps=20,
    learning_rate=2e-4,
    weight_decay=0.001,
    warmup_steps=10,
    optim="paged_adamw_32bit",
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

print("Starting training...")
trainer.train()

## 9. Evaluate on test set

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get model predictions on test set
def get_predictions(model, tokenizer, test_examples, max_new_tokens=10):
    predictions = []
    for ex in test_examples:
        inputs = tokenizer(ex["prompt"], return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
        )
        pred_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        # Extract first word (BUY, SELL, or HOLD)
        pred_action = pred_text.split()[0] if pred_text else "HOLD"
        predictions.append(pred_action)
    return predictions

print("Generating predictions on test set...")
pred_actions = get_predictions(model, tokenizer, test_data)
true_actions = [ex["response"] for ex in test_data]

# Compute metrics
accuracy = accuracy_score(true_actions, pred_actions)
print(f"\nTest Set Results:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Predictions: {pred_actions}")
print(f"  True labels: {true_actions}")

# Show sample predictions
for i in range(min(3, len(test_data))):
    print(f"\nSample {i+1}:")
    print(f"  True: {true_actions[i]}, Predicted: {pred_actions[i]}")

## 10. Save LoRA adapters for download

In [ ]:
# Save adapters
model.save_pretrained("models/checkpoints/xauusd_lora_final")
tokenizer.save_pretrained("models/checkpoints/xauusd_lora_final")

print("LoRA adapters saved to models/checkpoints/xauusd_lora_final/")

# Download
from google.colab import files
import shutil

# Zip the adapters for download
shutil.make_archive("xauusd_lora_adapters", "zip", "models/checkpoints/xauusd_lora_final")
files.download("xauusd_lora_adapters.zip")

print("Downloaded xauusd_lora_adapters.zip")

## 11. (Optional) Test inference with loaded adapters

In [ ]:
from peft import AutoPeftModelForCausalLM

# Merge adapters back to base model (optional, for simpler inference)
merged_model = model.merge_and_unload()
merged_model.save_pretrained("models/xauusd_merged")

print("Merged model saved to models/xauusd_merged/")
print("\nTraining complete! Next steps:")
print("1. Download the adapters ZIP file")
print("2. Extract to your local machine: models/checkpoints/xauusd_lora_final/")
print("3. In your inference code, load adapters using PEFT and the base model")